In [1]:
!pip cache purge

Files removed: 0
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [3]:
!pip install --quiet numpy pandas huggingface_hub transformer_lens torch tqdm transformers python-dotenv

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 6.4 MB/s eta 0:00:00


In [4]:
import glob
import os
import shutil
import numpy as np
import json
import pandas as pd
from huggingface_hub import login, snapshot_download, hf_hub_download
from transformer_lens import HookedTransformer
import torch
from tqdm.auto import tqdm
import gc

torch.set_grad_enabled(False)

torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [5]:
load_dotenv()
login(os.environ["HF_TOKEN"])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


# Loading the model

In [6]:
MODEL_NAME = "Qwen/Qwen3-8B"

In [7]:
from transformers import AutoConfig

In [8]:
config = AutoConfig.from_pretrained(MODEL_NAME, dtype=torch.bfloat16, device="cuda")
config.max_position_embeddings = 16000

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

In [ ]:
model = HookedTransformer.from_pretrained(MODEL_NAME, dtype=torch.bfloat16, device="cuda")
n_layers = model.cfg.n_layers
if model.tokenizer.pad_token is None:
    model.tokenizer.pad_token = model.tokenizer.eos_token

/tmp/ipykernel_574/3770297171.py:1: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained(MODEL_NAME)


model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

# Loading the data

In [ ]:
#json_path = "rule_clause_only_judgment_rules_expanded.verified.json"

In [ ]:
#with open(json_path) as file:
#    data = json.load(file)

In [ ]:
#data['pairs'][0].keys()

In [ ]:
LANG = 'yo'

In [ ]:
dpath = f"/workspace/data/{LANG}/test.jsonl"
with open(dpath) as file:
    data = [json.loads(line) for line in file]

In [ ]:
data[0].keys(), data[0]['language'], {i['language'] for i in data}

In [ ]:
new_data = [{**i, "system": i['context'] + ' ' + i['rule_clause'] + '.'}
            for i in data
           ]

In [ ]:
new_data[0].keys(), new_data[0]['system']

# Running inference

## Tokenising

In [ ]:
df = pd.DataFrame([{'id': i['id'], 'system': i['system'], 'query': i['user_query']} for i in new_data])

In [ ]:
df['system'].unique().shape, df['query'].unique().shape, df.shape

In [ ]:
df['prompt'] = df['system'].str.cat(df['query'], sep=' ', na_rep='')

In [ ]:
df['prompt'].unique().shape

In [ ]:
df['raw_id'] = df['id'].apply(lambda x: x.rsplit('_', maxsplit=1)[0])

In [ ]:
df['raw_id'].unique().shape

In [ ]:
unique_ids = df.groupby('prompt')['id'].first().to_frame().reset_index()['id'].tolist()

In [ ]:
df.loc[df['id'].isin(unique_ids)].shape, df['prompt'].shape, df['prompt'].unique().shape

In [ ]:
unique_df = df.loc[df['id'].isin(unique_ids)].copy()

In [ ]:
mapped_ids = unique_df['id'].tolist()
batch_conversations = [
    [
        {"role": "system", "content": system},
        {"role": "user", "content": query}
    ]
    for _, system, query in unique_df[['id', 'system', 'query']].values
]

In [ ]:
chat_prompt_strings = [
    model.tokenizer.apply_chat_template(
    conv,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
    ) for conv in batch_conversations
]

In [ ]:
len(chat_prompt_strings), len(set(chat_prompt_strings)), chat_prompt_strings[:3]

In [ ]:
NUM_SAMPLES = 3
expanded_mapped_ids = [ix for ix in mapped_ids for _ in range(NUM_SAMPLES)]
expanded_prompts = [prompt for prompt in chat_prompt_strings for _ in range(NUM_SAMPLES)]

In [ ]:
expanded_mapped_ids[:5]

## Generating

In [ ]:
BATCH_SIZE = 16
MAX_NEW_TOKENS = 768
all_generated_responses = []

In [ ]:
%%time
for i in tqdm(range(0, len(expanded_prompts), BATCH_SIZE), desc="Generating Batches"):
    batch_prompts = expanded_prompts[i : i + BATCH_SIZE]
    batch_tokens = model.to_tokens(batch_prompts, prepend_bos=False)

    # Run the generation block
    output_tokens = model.generate(
        batch_tokens,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=1.,
        top_p=0.9,
        prepend_bos=False,
        verbose=False
    )

    prompt_length = batch_tokens.shape[1]
    for row_tokens in output_tokens:
        new_response_tokens = row_tokens[prompt_length:]
        decoded_text = model.tokenizer.decode(new_response_tokens, skip_special_tokens=True)
        all_generated_responses.append(
            decoded_text.strip())

    del batch_tokens, output_tokens
    torch.cuda.empty_cache()
    if i % (100 * BATCH_SIZE) == 0:
        with open("rule_clause_only_yo_checkpoint.json", "w") as f:
            json.dump(
                [{"id": ix, "rule_clause_only_response": t}
                 for ix, t in zip(expanded_mapped_ids, all_generated_responses)],
                f)

In [ ]:
generated = [
    {'rule_clause_only_response': text, 'sample_idx': i % NUM_SAMPLES, 'id': ix}
    for i, (ix, text) in enumerate(zip(expanded_mapped_ids, all_generated_responses))
]

## Pushing to HF

In [ ]:
from huggingface_hub import HfApi

REPO_PATH = f"Qwen/Qwen3-8B/rule_clause_inference_only/rule_clause_inference_only_{LANG}.json"

In [ ]:
REPO = "crosslingual-rule-following/prism-model-activations"
OUT_PATH = "rule_clause_only_generated_responses.json"

In [ ]:
with open(OUT_PATH, "w") as f:
    json.dump(generated, f)

In [ ]:
HfApi().upload_file(path_or_fileobj=OUT_PATH, path_in_repo=REPO_PATH,
                    repo_id=REPO, repo_type="dataset")

In [ ]:
del data
del df
del unique_df
del expanded_prompts
del generated

gc.collect()